In [1]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

In [ ]:
def load_data(path):
    return pd.read_csv(path)

df_printer = load_data("printer_data 2(in).csv")
df_service = load_data("service_pack_quote 1(in).csv")

df_printer_raw = df_printer.copy()
df_service_raw = df_service.copy()

In [ ]:
def explore(df):
    print("Shape:", df.shape)
    print(df.head())
    print("Nulls:")
    print(df.isnull().sum())
    df.info()
    print()
    print(df.describe())

print("Printer Data")
explore(df_printer)
print()
print("Service Pack Data")
explore(df_service)

In [ ]:
def explore_service_details(df):
    print("total rows:", len(df))
    print("unique guids:", df['guid'].nunique())
    null_prices = df[df['Price'].isnull()]
    print("Rows having null Price:", len(null_prices))
    print(null_prices[['guid','Device Product Number']].head(10))
    print()
    has_hash = df[df['Device Product Number'].str.contains('#', na=False)]
    print("Device Product Numbers with # char:", len(has_hash))

explore_service_details(df_service)

In [ ]:
def clean_col_names(df):
    df.columns = df.columns.str.strip().str.lower().str.replace(" ", "_")
    return df

def clean_price(col):
    col = (col.str.strip().str.replace(",", "", regex=False).str.replace("$", "", regex=False))
    return pd.to_numeric(col, errors="coerce")

def strip_string_cols(df):
    for c in df.select_dtypes(include=['str','object']).columns:
        df[c] = df[c].str.strip()
    return df

In [ ]:
def clean_printer(df):
    df = clean_col_names(df)
    df['listpricecurrent'] = clean_price(df['listpricecurrent'])
    df = strip_string_cols(df)
    df = df.drop_duplicates()
    return df

def clean_service(df):
    df = clean_col_names(df)
    df['price'] = clean_price(df['price'])
    df = df.drop_duplicates()
    return df

df_printer = clean_printer(df_printer_raw.copy())
df_service = clean_service(df_service_raw.copy())

print("Printer:", df_printer.shape)
print(df_printer.dtypes)
print()
print("Service:", df_service.shape)
print(df_service.dtypes)

In [ ]:
def split_dpi(df):
    dpi = df['printdpi'].str.upper()
    parts = dpi.str.split('X')
    df['printdpi_x'] = parts.str[0].str.strip()
    df['printdpi_y'] = parts.str[1]
    df.loc[df['printdpi_y'].isna(), 'printdpi_y'] = df.loc[df['printdpi_y'].isna(), 'printdpi_x']
    df['printdpi_y'] = df['printdpi_y'].str.strip()
    df['printdpi_x'] = pd.to_numeric(df['printdpi_x'], errors='coerce').fillna(0).astype(float)
    df['printdpi_y'] = pd.to_numeric(df['printdpi_y'], errors='coerce').fillna(0).astype(float)
    df = df.drop(columns=['printdpi'])
    return df

df_printer = split_dpi(df_printer)
print(df_printer)

In [ ]:
def merge_data(df_printer, df_service):
    df = pd.merge(df_printer, df_service, on='guid', how='inner')
    print(" shape:", df.shape)
    print("Unique guids:", df['guid'].nunique())
    return df

df = merge_data(df_printer, df_service)
df.head()

In [ ]:
def calc_stats(df, group_col, val_col):
    grouped = df.groupby(group_col)[val_col]
    result = grouped.agg(['mean','median'])
    result['mode'] = grouped.apply(lambda x: x.mode().iloc[0] if len(x.mode())>0 else np.nan)
    return result

def compute_and_save_stats(df):
    device_stats = calc_stats(df, 'devicetype', 'listpricecurrent')
    service_stats = calc_stats(df, 'devicetype', 'price')
    stats_table = pd.concat([device_stats, service_stats], axis=1,keys=['device_price','service_price'])
    print(stats_table)
    stats_table.to_csv('devicetype_wise_stats.csv')
    print("\nsaved")

compute_and_save_stats(df)

In [ ]:
def check_nulls_and_zeroes(df):
    print("Null counts")
    print(df.isnull().sum())
    print()
    print("Zero Counts")
    num_cols = df.select_dtypes(include=[np.number]).columns
    for c in num_cols:
        zeros = (df[c]==0).sum()
        if zeros > 0:
            print(c, "has", zeros, "zeroes")
    return num_cols

num_cols = check_nulls_and_zeroes(df)

In [ ]:
def fill_na(df, col):
    med = df[col].median()
    print("imputing", col, "NA with median:", med)
    df[col] = df[col].fillna(med)
    return df

def impute_all_nas(df, num_cols):
    for c in num_cols:
        if df[c].isnull().sum() > 0:
            df = fill_na(df, c)
    print("NAs after imputation:")
    print(df.isnull().sum())
    return df

df = impute_all_nas(df, num_cols)

In [ ]:
def get_outlier_count(df, col):
    q1 = df[col].quantile(0.25)
    q3 = df[col].quantile(0.75)
    iqr = q3 - q1
    low = q1 - 1.5*iqr
    high = q3 + 1.5*iqr
    count = ((df[col]<low) | (df[col]>high)).sum()
    print(col, ":", count, "outliers")

def detect_outliers(df, num_cols):
    for c in num_cols:
        get_outlier_count(df, c)

detect_outliers(df, num_cols)

In [ ]:
def plot_boxplots(df, cols):
    for col in cols:
        plt.figure(figsize=(6, 4))
        plt.boxplot(df[col].dropna())
        plt.title(col)
        plt.tight_layout()
        plt.show()

plot_boxplots(df, ['listpricecurrent', 'price', 'speedmono', 'speedcolor','standardinputcapacity', 'maximuminputcapacity'])

In [ ]:
def sanity_checks(df):
    print("total rows:", len(df))
    print("unique guids:", df['guid'].nunique())
    print("negative device prices:", (df['listpricecurrent']<0).sum())
    print("negative service prices:", (df['price']<0).sum())
    print("zero device prices:", (df['listpricecurrent']==0).sum())
    print("zero service prices:", (df['price']==0).sum())
    print("device_product_numbers with # char:", df['device_product_number'].astype(str).str.contains('#').sum())
    print()
    print(df['make'].value_counts())
    print()
    print(df['devicetype'].value_counts())

sanity_checks(df)

In [ ]:
def check_high_correlation(corr, threshold=0.5):
    cols = corr.columns
    for i in range(len(cols)):
        for j in range(i+1, len(cols)):
            val = corr.iloc[i, j]
            if abs(val) > threshold:
                print(cols[i], "& ", cols[j], ":", round(val, 4))

def correlation_analysis(df, num_cols):
    corr = df[num_cols].corr()
    print(corr)
    print()
    check_high_correlation(corr)
    return corr

corr = correlation_analysis(df, num_cols)

In [ ]:
def plot_heatmap(corr):
    plt.figure(figsize=(10, 8))
    sns.heatmap(corr, annot=True, fmt='.3f', cmap='coolwarm')
    plt.title('Correlation Heatmap')
    plt.tight_layout()
    plt.show()

plot_heatmap(corr)

In [ ]:
def check_redundant(df):
    print("guid unique:", df['guid'].nunique())
    print("device_product_number unique:", df['device_product_number'].nunique())
    print()
check_redundant(df)

In [ ]:
def prepare_model_data(df):
    cat_cols = df.select_dtypes(include=['str','object']).columns.tolist()
    cat_cols = [c for c in cat_cols if c not in ['guid','device_product_number']]
    bool_cols = df.select_dtypes(include=['bool']).columns.tolist()
    df_model = pd.get_dummies(df, columns=cat_cols, drop_first=True, dtype=int)
    for c in bool_cols:
        df_model[c] = df_model[c].astype(int)
    df_model = df_model.drop(columns=['guid','device_product_number'], errors='ignore')
    print("shape:", df_model.shape)
    print("Columns:", df_model.columns.tolist())
    return df_model

df_model = prepare_model_data(df)

In [ ]:
def split_data(df_model, target='price', test_size=0.30, seed=42):
    y = df_model[target]
    X = df_model.drop(columns=[target])
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=test_size, random_state=seed)
    print("train:", X_train.shape[0], "test:", X_test.shape[0])
    return X, X_train, X_test, y_train, y_test

X, X_train, X_test, y_train, y_test = split_data(df_model)

In [ ]:
def train_and_evaluate(X_train, X_test, y_train, y_test):
    model = LinearRegression()
    model.fit(X_train, y_train)
    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)
    print("Train R2:", round(r2_score(y_train, y_train_pred), 4))
    print("Test R2:", round(r2_score(y_test, y_test_pred), 4))
    print("Train RMSE:", round(np.sqrt(mean_squared_error(y_train, y_train_pred)), 2))
    print("Test RMSE:", round(np.sqrt(mean_squared_error(y_test, y_test_pred)), 2))
    return model, y_train_pred, y_test_pred

model, y_train_pred, y_test_pred = train_and_evaluate(X_train, X_test, y_train, y_test)

In [ ]:
def show_coefficients(model, X):
    coeff = pd.DataFrame({'feature': X.columns, 'coefficient': model.coef_})
    coeff = coeff.sort_values('coefficient', ascending=False).reset_index(drop=True)
    print("intercept:", round(model.intercept_, 4))
    print(coeff)

show_coefficients(model, X)

In [ ]:
def plot_actual_vs_predicted(y_test, y_test_pred):
    plt.figure(figsize=(8,6))
    plt.scatter(y_test, y_test_pred, alpha=0.5)
    plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--')
    plt.xlabel("Actual")
    plt.ylabel("Predicted")
    plt.title("Actual vs Predicted")
    plt.show()

def plot_residuals(y_test, y_test_pred):
    residuals = y_test - y_test_pred
    plt.figure(figsize=(8,5))
    plt.scatter(y_test_pred, residuals, alpha=0.5, color='coral')
    plt.axhline(y=0, color='black', linestyle='--')
    plt.xlabel("Predicted")
    plt.ylabel("Residuals")
    plt.title("Residual Plot")
    plt.show()

plot_actual_vs_predicted(y_test, y_test_pred)
plot_residuals(y_test, y_test_pred)